# MimicGuard v2 / 2: stream-level fixed false-alarm protocol and feature-oracle headroom (E1)

Simulation-only. All detectors, including CUSUM-wrapped ones, are thresholded so that the probability of any alarm in a 100-window normal stream equals p (0.05 and 0.01), using hundreds of normal calibration streams, and the realised rate is checked on separate normal streams with exact intervals. Reports Pd, detection delay, paired bootstrap AE-vs-classical differences with an equivalence verdict, and the gap to a supervised oracle on the six features. Also repeats a cell with correlated fading.

**Kaggle settings:** Accelerator = **None (CPU)**; Internet = **Off** (nothing is downloaded; no token or secret is needed);
no dataset needs to be added. Run with *Save Version -> Save & Run All (Commit)* for the long runs so the output is kept.
Set `MODE` in the config cell: `"smoke"` = a couple of minutes, to check that everything works; `"medium"` and `"full"` = the real runs.
Runs are resumable: if the session stops, run it again and finished units are skipped (the output folder `results_fixed_arl` must stay the same).

Estimated run time on 4 CPUs (not measured on Kaggle): smoke about 5 min; medium about 1-1.5 h; full about 3-5 h.

In [ ]:
# ---- 1. Setup: unpack the embedded code, check the environment ----
import base64, io, os, sys, zipfile, subprocess, platform, warnings
warnings.filterwarnings("ignore")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("mimicguard_run")
os.makedirs(WORK, exist_ok=True)
REPO = os.path.join(WORK, "mimicguard")
_B64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_B64))) as z:
    z.extractall(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.chdir(REPO)
try:
    import yaml
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
import numpy, scipy, sklearn
print("python", platform.python_version(), "| numpy", numpy.__version__, "| sklearn", sklearn.__version__, "| CPUs", os.cpu_count())
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
except Exception as e:
    print("torch missing:", e)

In [ ]:
# ---- 2. Unit tests ----
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-1800:], r.stderr[-800:])
assert r.returncode == 0, "unit tests failed: stop and send the output above"

In [ ]:
# ---- 3. Config ----
from mimicguard.config import load_config
from mimicguard.v2 import experiments as ex
from mimicguard.v2.protocol import hier_boot_diff, equivalence_verdict, cp_interval
import pandas as pd, numpy as np
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 300)
BASE = load_config(os.path.join(REPO, "experiments/configs/base.yaml"))
MODE = "medium"                    # "smoke" | "medium" | "full"
N_JOBS = os.cpu_count() or 1
OUT = os.path.join(WORK, "results_fixed_arl"); os.makedirs(OUT, exist_ok=True)
if MODE == "smoke":
    CELLS, SEEDS, SIZES = [("rayleigh", 20)], [0, 1], dict(n_train=12, n_val=8, n_cal=20, n_chk=20, n_test=10, n_otrain=12, H=40, n_test_win=80, onset=40)
    RHO_CELLS, RHO_SEEDS = [], []
elif MODE == "medium":
    CELLS, SEEDS, SIZES = [(f, 20) for f in ("none", "rician", "rayleigh")], [0, 1, 2], ex.FULL
    RHO_CELLS, RHO_SEEDS = [("rayleigh", 20)], [0, 1]
else:
    CELLS, SEEDS, SIZES = [(f, s) for f in ("none", "rician", "rayleigh") for s in (15, 20, 25)], [0, 1, 2, 3, 4], ex.FULL
    RHO_CELLS, RHO_SEEDS = [("rayleigh", 20)], [0, 1, 2]
print(MODE, CELLS, SEEDS, SIZES)

In [ ]:
# ---- 4. E1: stream-level fixed false-alarm protocol (alpha = 0.5), with a supervised feature-space oracle ----
df = ex.run_e1(BASE, CELLS, SEEDS, OUT, sizes=SIZES, n_jobs=N_JOBS, rho=0.0)
df.to_csv(os.path.join(OUT, "e1_rows.csv"), index=False)
print(len(df), "rows")

In [ ]:
# ---- 5. Robustness: correlated fading between windows (AR(1), rho = 0.9 and 0.99), Rayleigh 20 dB ----
rho_rows = []
for rho in (0.9, 0.99):
    if RHO_CELLS:
        rho_rows.append(ex.run_e1(BASE, RHO_CELLS, RHO_SEEDS, OUT, sizes=SIZES, n_jobs=N_JOBS, rho=rho))
dfr = pd.concat(rho_rows) if rho_rows else pd.DataFrame()
if len(dfr): dfr.to_csv(os.path.join(OUT, "e1_rho_rows.csv"), index=False)
print(len(dfr), "rows")

In [ ]:
# ---- 6. Analysis A: does the false-alarm control hold? (stream-level, p = 0.05 and 0.01) ----
n_chk = int(SIZES["n_chk"])
def pooled_fa(d):
    k = int(round((d.fa_chk * n_chk).sum())); n = n_chk * len(d); lo, hi = cp_interval(k, n)
    return pd.Series({"realised_fa": k / n, "ci_lo": lo, "ci_hi": hi, "legacy_window_far": d.legacy_far.mean()})
fa = df.groupby(["p", "detector", "fading", "snr_db"]).apply(pooled_fa).reset_index()
print("=== realised stream-level false-alarm rate on held-out normal streams (target = p) ===")
print(fa[fa.p == 0.05].pivot_table(index="detector", columns=["fading", "snr_db"], values="realised_fa").round(3))
bad = fa[fa.p < fa.ci_lo]
print("detectors/cells where the target p lies BELOW the 95% interval (control failed):", len(bad))
print(bad[["p", "detector", "fading", "snr_db", "realised_fa", "ci_lo", "ci_hi"]].round(3).to_string(index=False) if len(bad) else "none")
print("=== legacy per-window false-alarm rate (threshold at 1% of validation windows), pre-onset test windows; target 0.01 ===")
print(fa[fa.p == 0.05].pivot_table(index="detector", columns=["fading", "snr_db"], values="legacy_window_far").round(3))
fa.to_csv(os.path.join(OUT, "false_alarm_audit.csv"), index=False)

In [ ]:
# ---- 7. Analysis B: detection at the fixed stream-level false-alarm rate (mean over seeds) ----
g = df[df.p == 0.05].groupby(["detector", "fading", "snr_db"])[["pd", "cadd", "auc", "legacy_pd"]].mean().round(3)
for m in ("pd", "cadd", "auc"):
    print(f"=== {m} at p = 0.05 ===")
    print(g[m].unstack(["fading", "snr_db"]))
print("=== legacy per-window Pd (old protocol) for comparison ===")
print(g["legacy_pd"].unstack(["fading", "snr_db"]))
g.to_csv(os.path.join(OUT, "detection_summary.csv"))

In [ ]:
# ---- 8. Analysis C: AE vs the others, paired hierarchical bootstrap over streams and seeds (Pd, p = 0.05) ----
from mimicguard.v2.experiments import load_detect_arrays
MARGIN = 0.05   # equivalence margin in detection probability; fixed before looking at results
PAIRS = [("ae", "pca_L4"), ("ae", "lof"), ("ae", "ocsvm"), ("ae", "iforest"),
         ("ae_cusum", "pca_L4_cusum"), ("ae_cusum", "lof_cusum"), ("ae_cusum", "ocsvm_cusum"), ("ae_cusum", "iforest_cusum")]
res = []
for fading, snr in CELLS:
    dets = sorted({d for p in PAIRS for d in p})
    A = load_detect_arrays(OUT, 0.0, fading, snr, "a0.5", 0.05, dets, SEEDS)
    for a, b in PAIRS:
        d, lo, hi = hier_boot_diff(A[a], A[b], B=2000)
        res.append({"fading": fading, "snr_db": snr, "a": a, "b": b, "diff_pd": d, "ci_lo": lo, "ci_hi": hi,
                    "verdict": equivalence_verdict(lo, hi, MARGIN)})
res = pd.DataFrame(res); res.to_csv(os.path.join(OUT, "ae_vs_others_bootstrap.csv"), index=False)
print(res.round(3).to_string(index=False))
print("verdict: 'equivalent' = whole 95% CI inside +/-%.2f; 'different' = CI excludes 0; otherwise 'inconclusive'." % MARGIN)

In [ ]:
# ---- 9. Analysis D: headroom. How far is the best unsupervised detector from the supervised oracle? (Pd, p = 0.05) ----
t = df[df.p == 0.05].groupby(["detector", "fading", "snr_db"]).pd.mean().unstack("detector")
unsup = [c for c in t.columns if not c.startswith("oracle") and c not in ("energy", "cusum")]
out = pd.DataFrame({"best_unsupervised_pd": t[unsup].max(axis=1), "best_is": t[unsup].idxmax(axis=1),
                    "oracle_pd": t["oracle_feat"], "oracle_cusum_pd": t["oracle_feat_cusum"]})
out["gap_to_oracle"] = out[["oracle_pd", "oracle_cusum_pd"]].max(axis=1) - out["best_unsupervised_pd"]
print(out.round(3)); out.to_csv(os.path.join(OUT, "headroom_features.csv"))
if len(dfr):
    print("=== correlated fading: realised stream-level false alarm (p = 0.05) and Pd ===")
    print(dfr[dfr.p == 0.05].groupby(["rho", "detector"])[["fa_chk", "pd", "auc"]].mean().round(3).unstack("rho"))

In [ ]:
# ---- last cell: package outputs ----
import shutil
zp = shutil.make_archive(os.path.join(WORK, "mimicguard_v2_fixed_arl_results"), "zip", OUT)
print("results archive:", zp, f"({os.path.getsize(zp)/1e6:.1f} MB)")
print("Download it from the Output tab (after Save Version -> Save & Run All), then send it back.")